# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JunaidAhamed-7777/flyrank-machine-learning/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Audit capstone metrics the way the March 2026 FlyRank paper was audited: methodology questions on published findings, then grouped/time validation on our Week-5 model.


## 1. Two paper findings + my methodology questions

*Source: `docs/flyrank-seo-research-march-2026.pdf` (text extracted with pypdf; readable).*

**Finding A (Finding #1 — The Anatomy of Growing Content).** In the portfolio comparison of rising versus falling impression trends, declining pages average about **2.3K words** and **230 days** of age versus **3.2K words** and **184 days** for growing pages — the paper frames this as older, thinner URLs losing momentum while still carrying impressions.

**Methodology question:** *Does the “up vs down” label use the same 30-day-versus-previous-30-day impression window as the operational refresh queue, and are word count and age measured strictly before that label window closes?* That matters because age and length are often updated after a refresh; if metadata or body length changed during the outcome window, the comparison mixes pre-decision structure with post-change content and overstates how much “thin/older” *precedes* decline.

**Finding B (Finding #3 — Click Capture by Position Tier).** The paper reports portfolio **weighted CTR** falling from about **0.423%** in the top-3 tier to **0.050%** in the deep tier, arguing that click capture compresses quickly as average position worsens.

**Methodology question:** *Are these tier CTRs computed from pooled clicks/impressions at portfolio scope, and does any holdout or appendix model reuse tier-level aggregates that were fit on the same pages being ranked?* Portfolio-level weighted CTR is a useful descriptive number; if a prioritisation model or subgroup claim reuses tier medians computed on the evaluation set, the validation design no longer supports generalisation to unseen sites or clients.

In [1]:
from pathlib import Path
import pypdf
pdf = Path('../../docs/flyrank-seo-research-march-2026.pdf')
text = ''.join((p.extract_text() or '') for p in pypdf.PdfReader(pdf).pages)
print(f'PDF readable: {len(text) > 10000} ({len(text)} chars)')
print('Sample:', text[500:900].replace('\n', ' ')[:200])


PDF readable: True (53356 chars)
Sample: 79,632 search impressions, 1,514,819  clicks,1,635,404 sessions, and 17,344 AI sessions. Rather than reproducing dashboard screenshots or internal table views, we tested portfolio-level questions abou


## 2. My model under an honest split (before/after)

*Re-run Week-5 RF: w05 holdout (before), LOO-client + time-aware (after). Precision@50 only; ranking within each evaluation frame.*
